## PILLAR 3: Permissions & Ownership

**Support context:** "Permission denied" is probably the single most common Linux support ticket across every domain — web servers, cron jobs, SSH, deploys, shared team directories. You already know how to _read_ `ls -l` output at a glance; now we go deep enough to _fix_ what you find.

**Reading `ls -l` fully, one more time, precisely**

In [ ]:
dennie@HP:~/linux-tut$ ls -l /etc/passwd /etc/shadow
-rw-r--r-- 1 root root   1426 Aug 21 21:33 /etc/passwd
-rw-r----- 1 root shadow  802 Aug 21 21:33 /etc/shadow

Break the permission string into 4 parts:

In [ ]:
-  rw-  r--  r--
│   │    │    │
│   │    │    └── OTHER: everyone else
│   │    └─────── GROUP: members of the file's group
│   └──────────── USER (owner): the file's owner
└──────────────── file type (- regular file, d directory, l symlink)

Each triplet is `r` (read), `w` (write), `x` (execute) or `-` (absent).

- **On a file**, `r`=view content, `w`=modify content, `x`=run it as a program.
- **On a directory**, the meanings shift subtly and matter a lot: `r`=list contents (`ls`), `w`=create/delete/rename entries inside it, `x`=**enter it** (`cd` into it, or access files inside by full path) even without `r`.

> **GOTCHA:** a directory with `x` but no `r` lets you `cd somefile/known_name.txt` if you know the exact filename, but `ls` inside it shows nothing. This trips people up constantly — "I can access the file directly but can't browse the folder" is not broken, it's `--x` on the directory working exactly as designed.

---

### Numeric (octal) notation

Each `rwx` triplet is really 3 bits: `r`=4, `w`=2, `x`=1. Add them up per triplet:

|Symbolic|Sum|Meaning|
|---|---|---|
|`rwx`|4+2+1=**7**|read+write+execute|
|`rw-`|4+2=**6**|read+write|
|`r-x`|4+1=**5**|read+execute|
|`r--`|4=**4**|read only|
|`---`|**0**|nothing|

So a permission string is just three digits, one per triplet (user/group/other):

|Numeric|Meaning|Typical use|
|---|---|---|
|`755`|`rwxr-xr-x`|executables, scripts, directories you want browsable|
|`644`|`rw-r--r--`|regular files — owner edits, everyone reads|
|`600`|`rw-------`|private files — SSH keys, credentials — owner only|
|`777`|`rwxrwxrwx`|everyone can do everything — **almost never correct**, treat as a red flag|
|`700`|`rwx------`|private directories — owner only|

**Developer bridge:** you already do this exact calculation every time you `chmod 600 id_rsa` before an SSH connection or set `644` on a static file for a web server — this is just naming the math behind numbers you've been typing on faith.

---

### `chmod` — change permissions

**SYNTAX:** `chmod [mode] file` — mode can be numeric (`755`) or symbolic (`u+x`, `go-w`).

In [ ]:
dennie@HP:~/linux-tut$ mkdir /tmp/permtest && cd /tmp/permtest
dennie@HP:~/linux-tut$ touch script.sh

In [ ]:
dennie@HP:/tmp/permtest$ ls -l script.sh
-rw-r--r-- 1 dennie dennie 0 Aug 25 18:25 script.sh

In [ ]:
dennie@HP:/tmp/permtest$ chmod 755 script.sh

In [ ]:
dennie@HP:/tmp/permtest$ ls -l script.sh
-rwxr-xr-x 1 dennie dennie 0 Aug 25 18:25 script.sh

symbolic form: add execute for the owner only:

In [ ]:
dennie@HP:/tmp/permtest$ chmod u+x script.sh

remove write from group and other:

In [ ]:
dennie@HP:/tmp/permtest$ chmod go-w script.sh

**Symbolic syntax:** `u`(user)/`g`(group)/`o`(other)/`a`(all), then `+`(add)/`-`(remove)/`=`(set exactly), then the permission letter(s).

**TICKET SCENARIO — "script won't execute":**

In [ ]:
./script.sh
bash: ./script.sh: Permission denied

In [ ]:
ls -l script.sh
-rw-r--r-- 1 dennie dennie 0 ... script.sh

**DIAGNOSIS PATH:** no `x` anywhere in the string → fix with `chmod +x script.sh` → retry. This is _the_ most common "why won't my script run" ticket, and it's always this.

---

### `chown` / `chgrp` — change owner and group

In [ ]:
dennie@HP:/tmp/permtest$ sudo chown root /tmp/permtest/script.sh
[sudo] password for dennie:

In [ ]:
dennie@HP:/tmp/permtest$ ls -l /tmp/permtest/script.sh
-rwxr-xr-x 1 root dennie 0 Aug 25 18:25 /tmp/permtest/script.sh

In [ ]:
dennie@HP:/tmp/permtest$ sudo chown dennie:dennie /tmp/permtest/script.sh

**SYNTAX:** `chown user:group file` (either side optional — `chown :group` changes only the group, equivalent to `chgrp`).
> **GOTCHA: requires sudo** unless you're root, because giving away ownership of your own files to someone else is a privileged operation (otherwise you could dodge disk quotas by dumping files on other users).

**TICKET SCENARIO — "web server can't read uploaded files":** a common deploy bug is files uploaded by one user (say `dennie`) but the web server process runs as a different user (`www-data`). If `www-data` isn't the owner and isn't in the file's group, and other-permission is `0`, the server gets `Permission denied` reading a file that's clearly sitting right there. **Fix path:** either `chown www-data:www-data` the file, or add the web server's user to a shared group and set group-read (`chmod 640` or add `g+r`).

---

### Testing file's ownership operations

In [ ]:
sudo -u www-data touch /var/www/kb/uploads/testfile && echo "write succeeded" && sudo -u www-data rm /var/www/kb/uploads/testfile

`sudo -u <user>` runs a command _as_ that specific user rather than as root, which is exactly the tool for "does this actually work for the account that matters," not just for you (`dennie`, who's in `sudo` and could write there for unrelated reasons).

---

### `umask` — the default permission mask for new files

In [ ]:
dennie@HP:/tmp/permtest$ umask
0022

**Concept:** new files don't get `777`/`666` by default even though that's the "everything on" baseline — `umask` _subtracts_ permissions from that baseline every time something's created. Standard Ubuntu umask is `022`, meaning: files start from `666` (`rw-rw-rw-`, directories can't execute by default so their baseline is `777`) and `022` strips write from group and other → new files land at `644`, new directories at `755`.

**Developer bridge:** you set this exact thing (implicitly) whenever a deploy script's output files come out world-writable and you have to remember to `chmod` them after — `umask` is the "set it once per session/user so you never have to remember" version.

In [ ]:
dennie@HP:/tmp/permtest$ touch newfile_default.txt

In [ ]:
dennie@HP:/tmp/permtest$ ls -l newfile_default.txt
-rw-r--r-- 1 dennie dennie 0 Aug 25 18:46 newfile_default.txt

In [ ]:
umask 077

In [ ]:
dennie@HP:/tmp/permtest$ touch newfile_private.txt

In [ ]:
dennie@HP:/tmp/permtest$ ls -l newfile_private.txt
-rw------- 1 dennie dennie 0 Aug 25 18:48 newfile_private.txt

**WHAT TO LOOK FOR:** the first file lands at `644` (default umask), the second at `600` (private umask) — same `touch` command, different result purely because of the shell's active umask. This is set per-session by default; changing it in `~/.bashrc` makes it permanent for that user (Pillar 9 territory, but worth knowing now).

---

### Special bits — setuid, setgid, sticky bit

We already saw one of these live: `sudo`/`su` are `-rwsr-xr-x` — that lowercase `s` in the owner slot is **setuid**.

|Bit|Symbol|On a **file**|On a **directory**|
|---|---|---|---|
|setuid|`s` in owner's `x` slot|run as the file's **owner**, not the invoking user|(rarely used on directories)|
|setgid|`s` in group's `x` slot|run as the file's **group**|new files/dirs created inside **inherit this directory's group**|
|sticky|`t` in other's `x` slot|(rarely used on regular files)|only the **owner** of a file can delete/rename it, even if others have write access to the directory|

**Numeric form:** a 4th leading digit — `4000`=setuid, `2000`=setgid, `1000`=sticky. `chmod 4755 file` sets setuid + `rwxr-xr-x`.

**Live proof — the sticky bit on `/tmp`:**

In [ ]:
dennie@HP:/tmp/permtest$ ls -ld /tmp
drwxrwxrwt 12 root root 4096 Aug 25 18:42 /tmp

**READ THE OUTPUT:** that trailing `t` (lowercase because owner-execute is also set — capital `T` would mean sticky-but-not-executable) is _why_ `/tmp` is safe to leave world-writable. Everyone can create files there (`rwx` for other), but the sticky bit means nobody can delete or rename **your** files out from under you, even though they technically have write access to the directory. Without sticky, any user could `rm` any other user's temp files.

**TICKET SCENARIO — shared team directory:**

In [ ]:
sudo mkdir /srv/teamshare
sudo chgrp devteam /srv/teamshare
sudo chmod 2775 /srv/teamshare

**DIAGNOSIS PATH:** `2775` = setgid + `rwxrwxr-x`. Setgid on the directory means every new file/folder created inside automatically belongs to the `devteam` group — without it, each user's new files would default to _their own_ primary group, and teammates would immediately hit "permission denied" writing to each other's files despite being in the "right" group. This exact setgid trick is the standard fix for "our shared project folder keeps breaking because of mismatched groups."

---

**TRY NOW — the full walkthrough**

In [ ]:
dennie@HP:/tmp/permtest$ mkdir -p /tmp/permtest && cd /tmp/permtest

In [ ]:
dennie@HP:/tmp/permtest$ touch script.sh && ls -l script.sh
-rwxr-xr-x 1 dennie dennie 0 Aug 25 18:52 script.sh

In [ ]:
dennie@HP:/tmp/permtest$ chmod +x script.sh && ls -l script.sh
-rwxr-xr-x 1 dennie dennie 0 Aug 25 18:52 script.sh

In [ ]:
dennie@HP:/tmp/permtest$ umask
0077

In [ ]:
dennie@HP:/tmp/permtest$ touch defaultfile.txt && ls -l defaultfile.txt
-rw------- 1 dennie dennie 0 Aug 25 18:55 defaultfile.txt

In [ ]:
dennie@HP:/tmp/permtest$ ls -ld /tmp
drwxrwxrwt 11 root root 4096 Aug 25 18:53 /tmp

## Breaking Down Your Output

**The surprising part:** `touch script.sh && ls -l script.sh` showed `-rwxr-xr-x` _before_ you even ran `chmod`. That's not umask, and it's not `touch` magically making things executable — it's a real gotcha worth locking in:

❌ Assuming `touch` on an **existing** file resets its permissions ✅ `touch` on a file that already exists only updates its **timestamps** — it never touches permissions or content. Your `/tmp/permtest/script.sh` already existed at `755` from the earlier `chmod 755 script.sh` example I walked through in the lesson text — so `touch` just bumped its mtime to `18:52`, permissions untouched. That's also why the _second_ `chmod +x` was a no-op — the execute bit was already set, `chmod +x` doesn't clear anything, it only adds. 💡 Why: `touch`'s actual job is "create if missing, otherwise just refresh timestamps" — permission-setting was never part of its purpose. 🔁 **Support rule:** never assume `touch somefile` gives you a clean-slate file if `somefile` might already exist. If you need a guaranteed-fresh permission state, `rm` first or use `chmod` explicitly — don't rely on `touch`.

**Quick clean demo to see a _real_ chmod change** (optional but satisfying):

In [ ]:
dennie@HP:~/linux-tut$ mkdir /tmp/permtest
dennie@HP:~/linux-tut$ touch /tmp/permtest/script.sh && ls -l /tmp/permtest/script.sh
-rw-r--r-- 1 dennie dennie 0 Aug 26 04:28 /tmp/permtest/script.sh

In [ ]:
dennie@HP:~/linux-tut$ chmod +x /tmp/permtest/script.sh && ls -l /tmp/permtest/script.sh
-rwxr-xr-x 1 dennie dennie 0 Aug 26 04:28 /tmp/permtest/script.sh

This time the first `ls -l` should show `-rw-------` (fresh file, your `umask 0077` in effect), and only _after_ `chmod +x` does the `x` actually appear — a genuine before/after.

**`umask` → `0077`, `defaultfile.txt` → `-rw-------` (600)** — the math checks out exactly: baseline for a new file is `666` (`rw-rw-rw-`), subtract umask `077` → `600`. Confirmed live.

**`ls -ld /tmp` → `drwxrwxrwt`** — sticky bit (`t`) still there, as expected — that's a system-wide setting, not something your `umask` touches.

---

## Quick Drill — Read These Cold

No terminal needed, just tell me what each one means (owner/group/other permissions, plus what any special bit does):

**Q1:** `-rwSr--r--`

**Q2:** `drwxrws---`

**Q3:** `-rwxr-xr-x` with `chmod 640` applied to it — what's the resulting string?

Q1: it's a file; owner can read, write; capital S in onwer's x slot is means setuid - runs as the file's owner not the invoking user but not executable; file's group - members of the file's group can read only, so do other.
Q2: it's a directory; owner - ls into it, add/update/delete and cd into it; dir's group members can ls it, add/update/delete content, s means run as the file's group, nothin can others.
Q3: it's a file; owner can read/write/execute it, its group members can read/execute, so do other, after applying chmod 640: -rw-r----- owner can read/write, group members can only read and nothing can others.

**Q1 — mostly right, one refinement:** ✅ You correctly read `rw` present, `S` = setuid, and correctly caught that the owner's execute bit is _absent_ (that's exactly why it's capital `S` not lowercase `s`). Group and other at `r--` — correct.

💡 One extra layer worth knowing: setuid **without** execute is a bit of an oddity in practice — setuid only does anything the moment the file is actually _run_, and a file with no execute bit can't be run at all. So this file has the setuid flag stored in its metadata, but it's functionally inert until someone also adds `+x`. Seeing capital `S` on a real file is often a red flag during a security review — someone set setuid intending to make a privileged binary, then either forgot `+x` or (more concerning) it was stripped afterward.

**Q2 — this one has the key mix-up worth catching:**

❌ "s means run as the file's group" (for the directory's group slot) ✅ On a **directory**, setgid does _not_ mean "runs as the group" — directories aren't executed, so that meaning doesn't apply. Setgid on a directory means: **any new file or subdirectory created inside automatically inherits this directory's group**, instead of defaulting to the creator's own primary group. 💡 Why: this is literally the `/srv/teamshare` example from the lesson — `chmod 2775 /srv/teamshare` uses exactly this bit so that when _any_ team member creates a file in there, it comes out owned by group `devteam` automatically, instead of each person's files ending up in their own personal group and immediately breaking access for teammates.

> 🔁 **Support rule:** the same bit symbol (`s`/`S`/`t`) means a genuinely different thing depending on whether it's sitting on a file or a directory — always check which one you're looking at before interpreting it. (Your file-type read of the `d` and the `rwx`/`rws`/`---` triplets themselves was all correct — just the _meaning_ of that group `s` needed the directory-specific version.)

**Q3 — fully correct.** `-rwxr-xr-x` → `chmod 640` → `-rw-r-----`. Good catch that numeric `chmod` **sets** the permissions outright rather than modifying relative to what was there before — the original `rwxr-xr-x` is completely irrelevant to the result once you use the numeric form. That's actually the key difference from symbolic `chmod u+x` (relative) vs `chmod 640` (absolute) — you've now seen both in action.